Config and rejected-records table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
CHK = "/Volumes/workspace/bronze/raw/_chk_silver"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER}._rejected_records (
  source_table STRING, failed_rules STRING, record STRING, rejected_ts TIMESTAMP)
""")

The reusable Silver function

In [0]:
def run_silver(name, transform, keys, rules):
    target = f"{SILVER}.{name}"

    def upsert(batch_df, batch_id):
        df = transform(batch_df)

        # 1) Data quality: list the rules each row fails
        checks = [F.when(~F.coalesce(F.expr(expr), F.lit(False)), F.lit(rule))
                  for rule, expr in rules.items()]
        df = df.withColumn("_failed", F.array_compact(F.array(*checks)))
        bad  = df.filter(F.size("_failed") > 0)
        good = df.filter(F.size("_failed") == 0).drop("_failed")

        # 2) Send rejected rows to the quarantine table
        (bad.select(
            F.lit(name).alias("source_table"),
            F.concat_ws(",", "_failed").alias("failed_rules"),
            F.to_json(F.struct(*[c for c in bad.columns if c != "_failed"])).alias("record"),
            F.current_timestamp().alias("rejected_ts"))
         .write.mode("append").saveAsTable(f"{SILVER}._rejected_records"))

        # 3) Deduplicate on business key, keeping the latest ingested row
        w = Window.partitionBy(*keys).orderBy(F.col("_ingest_ts").desc())
        good = good.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

        # 4) Create the table on first run, otherwise MERGE (upsert)
        if not spark.catalog.tableExists(target):
            good.write.format("delta").saveAsTable(target)
        else:
            cond = " AND ".join([f"t.{k} = s.{k}" for k in keys])
            (DeltaTable.forName(spark, target).alias("t")
               .merge(good.alias("s"), cond)
               .whenMatchedUpdateAll()
               .whenNotMatchedInsertAll()
               .execute())

    (spark.readStream.table(f"{BRONZE}.{name}")
        .writeStream
        .foreachBatch(upsert)
        .option("checkpointLocation", f"{CHK}/{name}")
        .trigger(availableNow=True)
        .start().awaitTermination())

Run it on orders

In [0]:
def t_orders(df):
    return df.select(
        F.trim("order_id").alias("order_id"),
        F.trim("customer_id").alias("customer_id"),
        F.lower(F.trim("order_status")).alias("order_status"),
        F.to_timestamp("order_purchase_timestamp").alias("order_purchase_ts"),
        F.to_timestamp("order_approved_at").alias("order_approved_ts"),
        F.to_timestamp("order_delivered_carrier_date").alias("order_delivered_carrier_ts"),
        F.to_timestamp("order_delivered_customer_date").alias("order_delivered_customer_ts"),
        F.to_timestamp("order_estimated_delivery_date").alias("order_estimated_delivery_ts"),
        "_ingest_ts", "_source_file")

rules_orders = {
    "order_id_not_null": "order_id IS NOT NULL",
    "customer_id_not_null": "customer_id IS NOT NULL",
    "purchase_ts_not_null": "order_purchase_ts IS NOT NULL",
    "valid_status": "order_status IN ('created','approved','invoiced','processing','shipped','delivered','unavailable','canceled')",
}

run_silver("orders", t_orders, ["order_id"], rules_orders)

Check the result

In [0]:
spark.table("workspace.silver.orders").printSchema()
display(spark.sql("SELECT COUNT(*) AS rows, COUNT(DISTINCT order_id) AS ids FROM workspace.silver.orders"))
display(spark.sql("SELECT * FROM workspace.silver._rejected_records"))